# OpenDub — ready-to-run Google Colab

Run the cells from top to bottom. Upload a short video or audio clip in the app, choose a target language, then download MP4, MP3, SRT, and VTT results. Whisper and NLLB model weights download on first use; Edge TTS needs internet. Colab sessions and free GPU availability are temporary and not guaranteed. The Gradio share link is public to anyone who has it, so do not process private media in a shared session.

In [ ]:
from pathlib import Path

REPO = Path('/content/PlayGround')
if not (REPO / 'app.py').exists():
    !git clone https://github.com/moonedition7-alt/PlayGround.git {REPO}
%cd /content/PlayGround

In [ ]:
# Install system media tools and the pinned project dependency ranges.
!apt-get -qq update
!apt-get -qq install -y ffmpeg libsndfile1
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!ffmpeg -version | head -n 1
!ffprobe -version | head -n 1

In [ ]:
# Configure inference for the current Colab runtime. Models are fetched lazily
# by the app and cached in the runtime's Hugging Face cache.
import os
import torch

has_gpu = torch.cuda.is_available()
os.environ['WHISPER_DEVICE'] = 'cuda' if has_gpu else 'cpu'
os.environ['WHISPER_COMPUTE_TYPE'] = 'float16' if has_gpu else 'int8'
os.environ['HOST'] = '0.0.0.0'
os.environ['PORT'] = '7860'
print(f"Runtime: {'GPU — ' + torch.cuda.get_device_name(0) if has_gpu else 'CPU'}")
print('Whisper weights download when the selected model is used; NLLB translation weights download on first translation.')

## Optional: preload the translation model

Skip this if you want to save Colab RAM/disk. Running it downloads the multi-gigabyte NLLB-200 model before starting the UI; the first translated job otherwise downloads it automatically.

In [ ]:
PRELOAD_TRANSLATION_MODEL = False
if PRELOAD_TRANSLATION_MODEL:
    from app import _get_translator
    _get_translator()
    print('Translation model is cached for this Colab session.')
else:
    print('Skipping model preload; the app will fetch models as needed.')

In [ ]:
from app import build_app

demo = build_app().queue(default_concurrency_limit=1, max_size=8)
demo.launch(server_name='0.0.0.0', server_port=7860, share=True, show_error=True)

## Notes

- Start with a 10–30 second clip and the Whisper `tiny` or `base` model on CPU. Larger models and background separation can exceed free-runtime memory or session limits.
- Optional background preservation uses Demucs and downloads additional weights.
- Speaker count rotates available synthesized voices between transcript turns; the app does not identify source speakers.
- Translation runs locally. Edge TTS sends translated text to Microsoft's service and requires an internet connection.
- Download results before the Colab runtime ends. Outputs and model cache in a free session are temporary. Stop the runtime when finished.